In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, losses

def build_eeai_model(n_radars=6, n_antennas=3, n_bins=120):
    # Calcoliamo i canali totali (6 * 3 = 18 per la baseline)
    input_channels = n_radars * n_antennas
    
    # --- INPUT LAYER ---
    # Trattiamo i dati come un'immagine 1D (1 x 120) con molti canali
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- FEATURE EXTRACTION (MobileNet Style) ---
    # 1. Prima convoluzione standard per estrarre i segnali base
    x = layers.Conv2D(16, (1, 5), padding='same', activation='relu', name="conv_base")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # Riduce a 60 bin

    # 2. Separable Convolution (Risparmio RAM critico per ESP32)
    x = layers.SeparableConv2D(32, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Riduce a 30 bin

    # 3. Global Average Pooling (Distrugge l'asse spaziale, salva la memoria)
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # 4. Layer denso comune (Rappresentazione astratta della stanza)
    common_feat = layers.Dense(32, activation='relu', name="features")(x)

    # --- MULTI-HEAD OUTPUT ---
    # Testa A: Coordinate (4 persone * 2 coord = 8 output)
    # Usiamo 'linear' perché le coordinate X,Y possono essere qualsiasi valore in metri
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)

    # Testa B: Presenza (4 persone = 4 output)
    # Usiamo 'sigmoid' per avere una probabilità tra 0 e 1 per ogni slot
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    model = models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_v1")
    return model

# Costruiamo il modello
model = build_eeai_model()

# --- DEFINIZIONE DELLA LOSS PERSONALIZZATA ---
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE) ma lo annulla 
    se la persona non è presente nella ground truth.
    """
    # Assumiamo che la maschera sia passata esternamente o gestita via pesi
    # Per semplicità qui usiamo una versione standard, 
    # ma in fase di training useremo model.compile(loss_weights=...)
    return losses.mean_squared_error(y_true, y_pred)

model.compile(
    optimizer='adam',
    loss={
        "coords_head": "mse",       # Errore quadratico per la posizione
        "mask_head": "binary_crossentropy" # Cross-entropy per la presenza (0/1)
    },
    loss_weights={
        "coords_head": 1.0, 
        "mask_head": 0.5            # Diamo più importanza alla precisione della posizione
    }
)

model.summary()

I0000 00:00:1778838336.531007   13549 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778838336.570035   13549 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778838337.804123   13549 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1778838338.743640   13549 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model: "EEAI_Net_v1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ radar_input         │ (None, 1, 120,    │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_base (Conv2D)  │ (None, 1, 120,    │      1,456 │ radar_input[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 1, 60, 16) │          0 │ conv_base[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sep_conv_1          │ (None, 1, 60, 32) │        592 │ pool_1[0][0]      │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 1, 30, 32) │          0 │ sep_conv_1[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 32)        │          0 │ pool_2[0][0]      │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features (Dense)    │ (None, 32)        │      1,056 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coords_head (Dense) │ (None, 8)         │        264 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_head (Dense)   │ (None, 4)         │        132 │ features[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,500 (13.67 KB)

 Trainable params: 3,500 (13.67 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
import os
import glob
import numpy as np
import tensorflow as tf

# ==============================================================================
# 1. IL GENERATORE DI DATI (Il nostro tubo intelligente)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=2, alpha=0.05, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        # Se siamo in training, mescoliamo l'ordine dei file a ogni epoca
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        # Dice a Keras quanti "blocchi" ci sono in totale
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        # Keras chiama questa funzione per chiedere il blocco numero 'idx'
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        X_batch, y_coords_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            # 1. Caricamento Dati Grezzi
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   # Shape: (T, 6, 3, 120, 2)
            people_xy = data['people_xy']   # Shape: (T, 4, 2)
            people_mask = data['people_mask'] # Shape: (T, 4)
            T = raw_iq.shape[0]             # Numero di frame in questo file
            
            # 2. Magnitudo e Reshape (il trucco 1D)
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            # Fonde 6 radar e 3 antenne in 18 canali, aggiunge la dimensione altezza=1
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            # 3. Decluttering EMA on-the-fly (in tempo reale)
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            for t in range(T):
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
            X_batch.append(decluttered)

            # 4. Formattazione Target
            # Appiattiamo le coordinate da (4, 2) a un vettore da 8 numeri
            y_coords_batch.append(people_xy.reshape(T, 8))
            y_mask_batch.append(people_mask)

        # Uniamo tutti i frame del batch in un unico grande tensore
        X = np.concatenate(X_batch, axis=0)
        Y_coords = np.concatenate(y_coords_batch, axis=0)
        Y_mask = np.concatenate(y_mask_batch, axis=0)

        # Restituiamo Input e un Dizionario con le due Teste di Output
        return X, {"coords_head": Y_coords, "mask_head": Y_mask}

    def on_epoch_end(self):
        # Mescola i file alla fine di ogni epoca per evitare che la rete impari a memoria
        if self.is_training:
            np.random.shuffle(self.file_paths)

# ==============================================================================
# 2. IL TEST DEI MOTORI (Sanity Check)
# ==============================================================================

# Cerca tutti i file .npz nella tua cartella data
# (Assicurati che il percorso sia corretto!)
tutti_i_file = glob.glob("dataset/data/*.npz")
print(f"Trovati {len(tutti_i_file)} file .npz in totale.")

if len(tutti_i_file) == 0:
    print("ERRORE: Non trovo i file. Controlla il percorso in glob.glob()")
else:
    # Per questo test, usiamo solo 4 file per non perdere tempo!
    test_files = tutti_i_file[:4]
    print(f"Uso {len(test_files)} file per il Sanity Check...")

    # Creiamo il tubo
    generatore_prova = EEAIDataGenerator(test_files, batch_size=2, alpha=0.05)

    # Riprendiamo il modello che avevamo compilato prima (assicurati di aver eseguito 
    # la cella con la funzione build_eeai_model() e model.compile() prima di questo blocco)
    
    print("\n--- INIZIO ADDESTRAMENTO DI PROVA (1 Epoca) ---")
    # Lanciamo il fit passando il generatore al posto di X_train e Y_train
    history = model.fit(
        generatore_prova,
        epochs=1,
        verbose=1
    )
    print("--- SANITY CHECK SUPERATO! IL MODELLO RESPIRA! ---")

/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Trovati 24 file .npz in totale.
Uso 4 file per il Sanity Check...

--- INIZIO ADDESTRAMENTO DI PROVA (1 Epoca) ---


I0000 00:00:1778838341.137197   13549 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1778838343.096418   13625 cpu_allocator_impl.cc:82] Allocation of 129600000 exceeds 10% of free system memory.
W0000 00:00:1778838343.189348   13605 cpu_allocator_impl.cc:82] Allocation of 115200000 exceeds 10% of free system memory.
W0000 00:00:1778838343.246413   13605 cpu_allocator_impl.cc:82] Allocation of 115200000 exceeds 10% of free system memory.
W0000 00:00:1778838343.289871   13600 cpu_allocator_impl.cc:82] Allocation of 115200000 exceeds 10% of free system memory.
W0000 00:00:1778838343.290004   13600 cpu_allocator_impl.cc:82] Allocation of 115200000 exceeds 10% of free system memory.


2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - coords_head_loss: 13.7170 - loss: 14.0310 - mask_head_loss: 0.6279
--- SANITY CHECK SUPERATO! IL MODELLO RESPIRA! ---


In [3]:
import os
import glob
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint

# 1. I tuoi indici per lo split (Bozza 1)
train_indices = [22, 0, 1, 2, 3, 10, 14, 16, 17, 18, 19, 21, 8, 9, 12, 5, 6, 4]
val_indices = [23, 7, 11, 13, 15, 20]

# 2. Recupero tutti i file
tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = []
val_files = []

# 3. Smistamento dei file in base al numero nel nome
for filepath in tutti_i_file:
    # Estraiamo il numero dal nome del file (es: da "window_000022.npz" a 22)
    basename = os.path.basename(filepath)
    num_str = basename.replace("window_", "").replace(".npz", "")
    
    try:
        num = int(num_str)
        if num in train_indices:
            train_files.append(filepath)
        elif num in val_indices:
            val_files.append(filepath)
    except ValueError:
        pass # Ignora file che non hanno numeri nel nome

print(f"File di Training trovati: {len(train_files)}")
print(f"File di Validation trovati: {len(val_files)}")

# 4. Inizializziamo i due Motori (Generatori)
# Batch size a 16 o 32 è standard.
BATCH_SIZE = 16 

train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=False)

# 5. Callback per salvare SOLO il modello quando migliora sulla Validation
checkpoint = ModelCheckpoint(
    "eeai_best_model.keras", 
    monitor="val_loss", 
    save_best_only=True, 
    verbose=1
)

# 6. FUOCO ALLE POLVERI! (Addestramento vero)
EPOCHS = 15 # Partiamo con 15 epoche per non aspettare ore

print("\n--- INIZIO ADDESTRAMENTO SERIO ---")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint], # Nessun Early Stopping, solo il salvataggio!
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

File di Training trovati: 18
File di Validation trovati: 6

--- INIZIO ADDESTRAMENTO SERIO ---
Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 674ms/step - coords_head_loss: 8.8061 - loss: 8.6793 - mask_head_loss: 0.6559
Epoch 1: val_loss improved from None to 7.66893, saving model to eeai_best_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 9s/step - coords_head_loss: 9.4187 - loss: 8.8092 - mask_head_loss: 0.6000 - val_coords_head_loss: 7.2918 - val_loss: 7.6689 - val_mask_head_loss: 0.7544
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - coords_head_loss: 3.3634 - loss: 4.4981 - mask_head_loss: 0.8486
Epoch 2: val_loss improved from 7.66893 to 6.78655, saving model to eeai_best_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 20s 18s/step - coords_head_loss: 4.3156 - loss: 6.1220 - mask_head_loss: 0.7711 - val_coords_head_loss: 6.3865 - val_loss: 6.7866 - val_mask_head_loss: 0.8000
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 676ms/step - coords_head_loss: 4.7157 - loss: 5.8421 - mask_head_loss: 0.6274
Epoch 3: val

Durante il setup, avevamo creato un modello "Multi-Head" (a due teste). Per questo Keras ti stampa 3 valori di loss diversi.
1. coords_head_loss (L'Errore di Posizione - MSE)
    Che cos'è: È il Mean Squared Error (Errore Quadratico Medio). Misura la distanza matematica tra le coordinate (X, Y) reali e quelle predette dalla rete.Il tuo risultato: All'epoca 15, la validazione (val_coords_head_loss) è arrivata a 3.35 .Il senso fisico: Essendo un errore al quadrato, per capire di quanti metri sta sbagliando in media la rete, devi farne la radice quadrata. Circa 1.83 metri. Significa che, in media, la rete posiziona la persona a circa 1.8 metri di distanza da dove si trova realmente. Può sembrare tanto in una stanza da 4.8m x 7.2m, ma per un modello microscopico addestrato per sole 15 epoche è un punto di partenza pazzesco!
2. mask_head_loss (L'Errore di Presenza - Binary Crossentropy)
    Che cos'è: Misura quanto la rete è "confusa" sul fatto che una persona esista o meno (0 o 1). Più è vicino a zero, più la rete è sicura e ci azzecca. Il tuo risultato: È scesa da  1.06  a  0.88 . È un buon calo, significa che sta iniziando a capire quando ignorare i riflessi fantasma.
3. loss (La Loss Totale)
    Che cos'è: È il "voto finale" che Keras usa per aggiornare i pesi.Come si calcola: Ricordi che avevamo impostato dei pesi? (loss_weights={"coords_head": 1.0, "mask_head": 0.5}). La Loss totale è un'equazione matematica esatta. Prendiamo i tuoi dati dell'Epoca 15 di validazione: 3.3504 + (0.5 x 0.8889) = 3.3504 + 0.44445 = 3.79485  infatti, Keras ha stampato esattamente val_loss: 3.7949. La matematica torna perfettamente.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Caricamento Modello
print("Caricamento del modello migliore...")
best_model = tf.keras.models.load_model("eeai_best_model.keras", compile=False)

# 2. Estrazione di UN INTERO BATCH di validazione (Tutti i 120 frame)
X_val_batch, y_val_dict = val_gen[0]

# Prendiamo l'intera sequenza temporale del primo file del batch
# Shape X_seq: (120, 1, 120, 18)
X_seq = val_gen.file_paths[0] # Solo per sapere il nome del file
X_test_all_frames = X_val_batch[0:120] 

# Estraiamo la Verità Assoluta (Ground Truth) per tutti i frame
Y_true_coords_all = y_val_dict["coords_head"][0:120]
Y_true_mask_all = y_val_dict["mask_head"][0:120]

# Pre-calcoliamo TUTTE le previsioni per non laggare durante lo slider
print("Calcolo delle previsioni per tutti i 120 frame in corso...")
preds_all = best_model.predict(X_test_all_frames, verbose=0)
Y_pred_coords_all = preds_all[0]
Y_pred_mask_all = preds_all[1]
print("Fatto! Usa gli slider qui sotto.")

# 3. Funzione di Disegno (Verrà richiamata ad ogni movimento dello slider)
def plot_radar(frame_idx, soglia):
    plt.figure(figsize=(8, 10))
    plt.xlim(0, 4.8)
    plt.ylim(0, 7.2)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.title(f"Test Interattivo - Frame: {frame_idx}/119 | Soglia: {soglia:.2f}", fontsize=14, fontweight='bold')
    plt.xlabel("X - Larghezza (metri)", fontsize=12)
    plt.ylabel("Y - Lunghezza (metri)", fontsize=12)

    rect = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=2, edgecolor='black', facecolor='whitesmoke')
    plt.gca().add_patch(rect)

    # Estraiamo i dati del singolo frame selezionato dallo slider
    y_true_coords = Y_true_coords_all[frame_idx].reshape(4, 2)
    y_true_mask = Y_true_mask_all[frame_idx]
    y_pred_coords = Y_pred_coords_all[frame_idx].reshape(4, 2)
    y_pred_mask = Y_pred_mask_all[frame_idx]

    # Tracciamo VERITÀ e PREVISIONI
    for i in range(4):
        # Ground Truth
        if y_true_mask[i] == 1.0:
            rx, ry = y_true_coords[i]
            plt.scatter(rx, ry, c='green', s=150, marker='o', edgecolors='black', label='Reale' if i==0 else "")
            
        # Predizione
        conf = y_pred_mask[i]
        if conf > soglia:
            px, py = y_pred_coords[i]
            # Se la previsione è oltre la soglia, la facciamo rossa (più è sicura, più è scura)
            alpha_val = max(0.3, conf) 
            plt.scatter(px, py, c='red', s=150, marker='X', edgecolors='black', alpha=alpha_val, label='Predizione' if i==0 else "")
            plt.text(px + 0.15, py, f"{conf*100:.0f}%", fontsize=10, color='red', weight='bold')

    # Legenda dinamica
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=12)
    
    plt.tight_layout()
    plt.show()

# 4. Creazione dell'Interfaccia Grafica
widgets.interact(
    plot_radar, 
    # Partiamo dal frame 20 per saltare lo schifo iniziale dell'EMA
    frame_idx=widgets.IntSlider(value=50, min=10, max=119, step=1, description='Frame:'),
    # Soglia da 0.1 a 0.99
    soglia=widgets.FloatSlider(value=0.75, min=0.1, max=0.99, step=0.05, description='Soglia:')
);

Caricamento del modello migliore...
Calcolo delle previsioni per tutti i 120 frame in corso...
Fatto! Usa gli slider qui sotto.


interactive(children=(IntSlider(value=50, description='Frame:', max=119, min=10), FloatSlider(value=0.75, desc…